In [ ]:
import pandas as pd
import numpy as np

In [ ]:
files = [
    "/content/IT Support Ticket Data.csv",
    "/content/customer_support_tickets_200k.csv",
]

In [ ]:
dfs = []
for f in files:
  temp_df = pd.read_csv(f)
  if "body" in temp_df.columns:
    temp_df = temp_df["body"]
  elif "issue_description" in temp_df.columns:
    temp_df = temp_df["issue_description"]
  elif "Body" in temp_df.columns:
    temp_df = temp_df["Body"]
  dfs.append(temp_df)

In [ ]:
df = pd.concat(dfs, ignore_index = True)

In [ ]:
df = df.to_frame(name = "description")

In [ ]:
df.head(5)

,description
0,"Dear Customer Support Team,I am writing to rep..."
1,"Dear Customer Support Team,I hope this message..."
2,"Dear Customer Support Team,I hope this message..."
3,"Dear Support Team,I hope this message reaches ..."
4,"Dear Customer Support,I hope this message reac..."


In [ ]:
df = df.dropna()

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
def clean_text(text, keep_punctuation=True, lemmatize=False, remove_stopwords=False):
    """
    Enhanced text cleaning with configurable options
    """
    original_text = text

    text = text.lower()

    text = re.sub(r"^(dear|hi|hello|hey)\s+.*?,", "", text, flags=re.IGNORECASE)

    polite_phrases = [
        r"i hope this message finds you well",
        r"i hope this message reaches you",
        r"i hope you are doing well",
        r"i am reaching out to",
        r"just writing to",
        r"i wanted to",
        r"i would like to",
        r"i am writing to (report|inform|ask|request)",
        r"please (find|see) attached",
        r"attached please find"
    ]

    for phrase in polite_phrases:
        text = re.sub(phrase, "", text, flags=re.IGNORECASE)

    signoffs = [
        r"thank you.*$", r"thanks.*$", r"regards.*$",
        r"best regards.*$", r"sincerely.*$", r"cheers.*$",
        r"appreciate your (help|assistance|time).*$",
        r"looking forward.*$"
    ]

    for signoff in signoffs:
        text = re.sub(signoff, "", text, flags=re.IGNORECASE)

    text = re.sub(r"\b\w+@\w+\.\w+\b", "", text)  # emails
    text = re.sub(r"\b\d{10,}\b", "", text)  # phone numbers
    text = re.sub(r"ticket\s*id:?\s*\w+", "", text, flags=re.IGNORECASE)  # ticket IDs

    if keep_punctuation:
        text = re.sub(r"[^\w\s\.\?\!]", "", text)
    else:
        text = re.sub(r"[^\w\s]", "", text)

    text = re.sub(r"\s+", " ", text).strip()

    if remove_stopwords or lemmatize:
        tokens = text.split()

        if remove_stopwords:
            stop_words = set(stopwords.words('english'))
            domain_words = {'not', 'no', 'but', 'however', 'issue', 'problem', 'error'}
            stop_words = stop_words - domain_words
            tokens = [word for word in tokens if word not in stop_words]

        if lemmatize:
            lemmatizer = WordNetLemmatizer()
            tokens = [lemmatizer.lemmatize(word) for word in tokens]

        text = ' '.join(tokens)

    if not text or len(text.split()) < 2:
        text = original_text.lower()
        text = re.sub(r"[^\w\s]", "", text)
        text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
df["cleaned_text"] = df["description"].apply(clean_text)

In [ ]:
df = df.drop_duplicates(subset=['cleaned_text'], keep='first')

print(f"Original rows: {len(df)}")
print(f"After dedup on cleaned_text: {len(df)}")

Original rows: 25047
After dedup on cleaned_text: 25047


In [ ]:
df["cleaned_text"]

,cleaned_text
0,a significant problem with the centralized acc...
1,well. request detailed information about the c...
2,. request clarification about the billing and ...
3,well. ask about the compatibility of your prod...
4,in good health. i am eager to learn more about...
...,...
29656,there seems to be a discrepancy in my billing ...
29657,request a refund for the recent charge.
29658,twofactor authentication codes are not being d...
29660,i am unable to access my account after enterin...


In [ ]:
df['cleaned_text'].to_csv("desc.csv", index = False)

In [ ]:
%%capture
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
model_name = "BAAI/bge-large-en-v1.5"

In [ ]:
embedder = SentenceTransformer(model_name)
embedder.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': True, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [ ]:
texts = df["cleaned_text"].tolist()

In [ ]:
embeddings = embedder.encode(
    texts,
    batch_size = 64,
    show_progress_bar = True,
    normalize_embeddings = True
)

Batches:   0%|          | 0/392 [00:00<?, ?it/s]

In [ ]:
%%capture
!pip install cudf-cu12 cuml-cu12 --extra-index-url=https://pypi.nvidia.com

In [ ]:
import cuml.cluster

In [ ]:
import cuml

umap_model = cuml.UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.0,
    metric='cosine'
)
reduced_embeddings = umap_model.fit_transform(embeddings)

hdb = cuml.cluster.HDBSCAN(
    min_cluster_size=30,
    min_samples=1,
    metric='euclidean',
    cluster_selection_epsilon=0.1,
    cluster_selection_method='eom'
)
labels_gpu = hdb.fit_predict(reduced_embeddings)

In [ ]:
!pip install lz4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.2 MB/s eta 0:00:00


In [ ]:
import umap
import joblib
import numpy as np


embeddings_cpu = embeddings

cpu_umap = umap.UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)
reduced_embeddings_cpu = cpu_umap.fit_transform(embeddings_cpu)

np.save("reduced_embeddings.npy", reduced_embeddings_cpu)
# joblib.dump(cpu_umap, "cpu_umap.pkl")
joblib.dump(cpu_umap, "cpu_umap_compressed.pkl", compress=('zlib', 3))


print(f"UMAP model saved. Reduced shape: {reduced_embeddings_cpu.shape}")

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP model saved. Reduced shape: (25047, 5)


In [ ]:
n_clusters = len(np.unique(labels_gpu)) - (1 if -1 in labels_gpu else 0)
print(f"GPU HDBSCAN: {n_clusters} clusters")

GPU HDBSCAN: 183 clusters


In [ ]:
unique, counts = np.unique(labels_gpu, return_counts=True)
n_clusters = len(unique) - (1 if -1 in unique else 0)
noise_count = counts[unique == -1][0] if -1 in unique else 0

print(f"Total clusters: {n_clusters}")
print(f"Noise / outliers: {noise_count} ({noise_count / len(labels_gpu) * 100:.1f}%)")
print(f"Largest 10 clusters sizes: {sorted(counts[unique != -1], reverse=True)[:10]}")
print(f"Small clusters (<20 tickets): {sum(counts < 20)}")

Total clusters: 183
Noise / outliers: 3858 (15.4%)
Largest 10 clusters sizes: [np.int64(738), np.int64(726), np.int64(587), np.int64(578), np.int64(555), np.int64(534), np.int64(444), np.int64(432), np.int64(423), np.int64(404)]
Small clusters (<20 tickets): 0


In [ ]:
import cuml
import cupy as cp

is_clustered = labels_gpu != -1
is_noise = labels_gpu == -1

knn = cuml.neighbors.KNeighborsClassifier(n_neighbors=1)
knn.fit(embeddings[is_clustered], labels_gpu[is_clustered])

noise_reassignments = knn.predict(embeddings[is_noise])

final_labels = labels_gpu.copy()
final_labels[is_noise] = noise_reassignments

noise_count = cp.sum(final_labels == -1)
print(f"Remaining noise: {noise_count}")

Remaining noise: 0


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def get_cluster_keywords(texts, labels, n_words=10):
    df = pd.DataFrame({'text': texts, 'label': labels})
    docs_per_class = df.groupby(['label'], as_index=False).agg({'text': ' '.join})

    count = CountVectorizer(stop_words='english').fit(docs_per_class.text)
    t = count.transform(docs_per_class.text).toarray()
    w = t.sum(axis=1)
    tf = np.divide(t.T, w)
    sum_t = t.sum(axis=0)
    idf = np.log(np.divide(len(docs_per_class), sum_t, out=np.zeros_like(sum_t, dtype=float), where=sum_t!=0))
    tf_idf = np.multiply(tf.T, idf)

    words = count.get_feature_names_out()
    top_n_words = {label: [words[i] for i in tf_idf[i_label].argsort()[-n_words:]]
                   for i_label, label in enumerate(docs_per_class.label)}
    return top_n_words

keywords = get_cluster_keywords(texts, final_labels)

In [ ]:
keywords

In [ ]:
import cupy as cp
import pandas as pd
from sklearn.cluster import AgglomerativeClustering


unique_labels = sorted([l for l in np.unique(final_labels) if l != -1])
centroids = cp.array([embeddings[final_labels == l].mean(axis=0) for l in unique_labels])

n_meta_clusters = 30
merger = AgglomerativeClustering(n_clusters=n_meta_clusters, metric='cosine', linkage='average')
meta_labels = merger.fit_predict(centroids.get())

cluster_to_meta_map = {old_idx: meta_idx for old_idx, meta_idx in zip(unique_labels, meta_labels)}

df['cluster_id'] = final_labels
df['meta_cluster_id'] = df['cluster_id'].map(cluster_to_meta_map)

df['meta_label_name'] = df['meta_cluster_id'].apply(lambda x: f"Meta-Group {x}")

In [ ]:
df.head(5)

,description,cleaned_text,cluster_id,meta_cluster_id,meta_label_name
0,"Dear Customer Support Team,I am writing to rep...",a significant problem with the centralized acc...,41,6,Meta-Group 6
1,"Dear Customer Support Team,I hope this message...",well. request detailed information about the c...,114,2,Meta-Group 2
2,"Dear Customer Support Team,I hope this message...",. request clarification about the billing and ...,31,23,Meta-Group 23
3,"Dear Support Team,I hope this message reaches ...",well. ask about the compatibility of your prod...,81,0,Meta-Group 0
4,"Dear Customer Support,I hope this message reac...",in good health. i am eager to learn more about...,126,2,Meta-Group 2


In [ ]:
df['meta_label_name'].value_counts()

,count
meta_label_name,
Meta-Group 6,4026
Meta-Group 10,3285
Meta-Group 4,3257
Meta-Group 0,2762
Meta-Group 2,2316
Meta-Group 12,2063
Meta-Group 11,1986
Meta-Group 3,1902
Meta-Group 23,1033


In [ ]:
from collections import Counter

In [ ]:
cluster_counts = Counter(labels_gpu)
sorted_clusters = sorted(cluster_counts.items(), key = lambda x: x[1], reverse = True)

In [ ]:
cluster_counts = df["meta_cluster_id"].value_counts()
top_clusters = cluster_counts.index.sort_values()

for cid in top_clusters:
    if cid == -1: continue

    cluster_df = df[df["meta_cluster_id"] == cid]
    cluster_size = len(cluster_df)

    if cluster_size == 0:
        continue

    print(f"\n--- Meta-Group {cid} ({cluster_size} tickets) ---")

    samples = cluster_df["cleaned_text"].sample(
        min(10, cluster_size),
        random_state=42
    )

    for i, text in enumerate(samples, 1):
        preview = str(text).replace('\n', ' ').strip()
        print(f"  {i}. {preview[:180]}...")


--- Meta-Group 0 (2762 tickets) ---
  1. i am seeking guidance on how to integrate salesforce crm with digital marketing strategies. could you provide specific details on the benefits this integration brings to customers?...
  2. the level of engagement is below the anticipated threshold....
  3. customer support brwe are encountering difficulties with our marketing campaigns which are underperforming due to inconsistencies in our digital strategies. recent algorithm change...
  4. bring to your attention a sudden decrease in social media engagement. this might be related to the recent algorithm modifications. despite tweaking our strategies for targeting and...
  5. i am reporting that our digital marketing campaigns are being disrupted due to connectivity software problems. it appears that the issues originated from modem failure related to f...
  6. the marketing firm is experiencing a decline in brand growth through various digital strategies. this may be due to the use of outdate

In [ ]:
cluster_names = {
    'Meta-Group 0': 'Digital Marketing Performance Issues',
    'Meta-Group 1': 'End-User Device Issues',
    'Meta-Group 2': 'System Configuration & Integration Issues',
    'Meta-Group 3': 'Marketing Strategy Requests',
    'Meta-Group 4': 'Data Breach & Unauthorized Access Incidents',
    'Meta-Group 5': 'Integration & API Requests',
    'Meta-Group 6': 'System Performance & Availability Issues',
    'Meta-Group 7': 'Product & Service Information Requests',
    'Meta-Group 8': 'Infrastructure & Operational Issues',
    'Meta-Group 9': 'Billing & Payment Inquiries',
    'Meta-Group 10': 'Data Processing & Reporting Issues',
    'Meta-Group 11': 'Investment Analytics & Optimization Requests',
    'Meta-Group 12': 'Data Security & Compliance Guidance',
    'Meta-Group 13': 'Analytics Tracking & Dashboard Issues',
    'Meta-Group 14': 'Billing Errors & Invoice Discrepancies',
    'Meta-Group 15': 'Database Performance & Connectivity Issues',
    'Meta-Group 16': 'Integration Requests (Project Management Tools)',
    'Meta-Group 17': 'Analytics Tools Integration & Usage Issues',
    'Meta-Group 18': 'Peripheral Device Issues (Printers & Accessories)',
    'Meta-Group 19': 'Application Usage & Configuration Issues',
    'Meta-Group 20': 'Network & Connectivity Issues',
    'Meta-Group 21': 'Analytics Tools & Integration Guidance',
    'Meta-Group 22': 'Security Tools Setup & Usage Guidance',
    'Meta-Group 23': 'Billing Errors & Payment Issues',
    'Meta-Group 24': 'Search & Indexing System Issues',
    'Meta-Group 25': 'Encryption & Data Protection Issues',
    'Meta-Group 26': 'System Requirements & Compatibility Guidance',
    'Meta-Group 27': 'General Assistance Requests',
    'Meta-Group 28': 'Integration & Implementation Instructions',
    'Meta-Group 29': 'Network & Router Connectivity Issues'
}

In [ ]:
import json

with open("meta_cluster_names.json", "w") as f:
  json.dump(cluster_names, f)

In [ ]:

final_cluster_names = {
    "Meta-Group 0": "Marketing & Digital Strategy",
    "Meta-Group 1": "Hardware & End-User Devices",
    "Meta-Group 2": "System Configuration & Integration",
    "Meta-Group 3": "Marketing & Digital Strategy",
    "Meta-Group 4": "Data Security & Compliance",
    "Meta-Group 5": "System Configuration & Integration",
    "Meta-Group 6": "System Performance & Availability",
    "Meta-Group 7": "Product & Service Information",
    "Meta-Group 8": "System Performance & Availability",
    "Meta-Group 9": "Billing, Payments & Subscriptions",
    "Meta-Group 10": "System Performance & Availability",
    "Meta-Group 11": "Financial Analytics & Investment Insights",
    "Meta-Group 12": "Data Security & Compliance",
    "Meta-Group 13": "System Performance & Availability",
    "Meta-Group 14": "Billing, Payments & Subscriptions",
    "Meta-Group 15": "System Performance & Availability",
    "Meta-Group 16": "System Configuration & Integration",
    "Meta-Group 17": "System Configuration & Integration",
    "Meta-Group 18": "Hardware & End-User Devices",
    "Meta-Group 19": "System Configuration & Integration",
    "Meta-Group 20": "Network & Connectivity",
    "Meta-Group 21": "System Configuration & Integration",
    "Meta-Group 22": "Data Security & Compliance",
    "Meta-Group 23": "Billing, Payments & Subscriptions",
    "Meta-Group 24": "System Performance & Availability",
    "Meta-Group 25": "Data Security & Compliance",
    "Meta-Group 26": "System Configuration & Integration",
    "Meta-Group 27": "General Assistance",
    "Meta-Group 28": "System Configuration & Integration",
    "Meta-Group 29": "Network & Connectivity"
  }

In [ ]:
import json

with open("final_cluster_names.json", "w") as f:
  json.dump(final_cluster_names, f)

In [ ]:
df["merged_category"] = df["meta_label_name"].map(final_cluster_names)

df["merged_category"] = df["merged_category"].fillna("Uncategorized / Rare Issues")

print(df["merged_category"].value_counts())

merged_category
System Performance & Availability            7884
Data Security & Compliance                   5545
Marketing & Digital Strategy                 4664
System Configuration & Integration           2832
Financial Analytics & Investment Insights    1986
Billing, Payments & Subscriptions            1352
Hardware & End-User Devices                   484
Network & Connectivity                        200
General Assistance                             61
Product & Service Information                  39
Name: count, dtype: int64


In [ ]:
import json
with open("merged_cluster_names.json", "w") as f:
    json.dump(final_cluster_names, f, indent=2)

In [ ]:
import pickle

with open('umap_model.pkl', 'wb') as f:
    pickle.dump(umap_model, f)

with open('knn_model.pkl', 'wb') as f:
    pickle.dump(knn, f)

with open('category_mapping.pkl', 'wb') as f:
    pickle.dump(final_cluster_names, f)

In [ ]:
import re

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = " ".join(text.split())
    return text

def classify_new_ticket(ticket_text):
    clean_text = preprocess_text(ticket_text)

    new_embedding = embedder.encode([clean_text], normalize_embeddings=True)

    raw_pred = knn.predict(new_embedding)
    cluster_id = int(raw_pred[0])

    meta_id = cluster_to_meta_map.get(cluster_id, -1)
    key = f"Meta-Group {meta_id}"
    category = final_cluster_names.get(key, "Uncategorized")

    return category

In [ ]:
clean = "Customer suggests adding dark mode to improve usability at night."
print(f"Clean Result: {classify_new_ticket(clean)}")

Clean Result: System Configuration & Integration


In [ ]:
import joblib

In [ ]:
joblib.dump(embeddings, 'best_embeddings.joblib')

['best_embeddings.joblib']

In [ ]:
import numpy as np
np.save("embeddings.npy", embeddings)

In [ ]:
df

,description,cleaned_text,cluster_id,meta_cluster_id,meta_label_name,merged_category
0,"Dear Customer Support Team,I am writing to rep...",a significant problem with the centralized acc...,41,6,Meta-Group 6,System Performance & Availability
1,"Dear Customer Support Team,I hope this message...",well. request detailed information about the c...,114,2,Meta-Group 2,System Configuration & Integration
2,"Dear Customer Support Team,I hope this message...",. request clarification about the billing and ...,31,23,Meta-Group 23,"Billing, Payments & Subscriptions"
3,"Dear Support Team,I hope this message reaches ...",well. ask about the compatibility of your prod...,81,0,Meta-Group 0,Marketing & Digital Strategy
4,"Dear Customer Support,I hope this message reac...",in good health. i am eager to learn more about...,126,2,Meta-Group 2,System Configuration & Integration
...,...,...,...,...,...,...
29656,There seems to be a discrepancy in my billing ...,there seems to be a discrepancy in my billing ...,43,14,Meta-Group 14,"Billing, Payments & Subscriptions"
29657,I would like to request a refund for the recen...,request a refund for the recent charge.,66,23,Meta-Group 23,"Billing, Payments & Subscriptions"
29658,Two-factor authentication codes are not being ...,twofactor authentication codes are not being d...,158,6,Meta-Group 6,System Performance & Availability
29660,I am unable to access my account after enterin...,i am unable to access my account after enterin...,42,6,Meta-Group 6,System Performance & Availability


In [ ]:
np.save("meta_cluster_ids.npy", df["meta_cluster_id"].values)

df["merged_category"].to_csv("merged_category.csv", index=False)

In [ ]:
desc = pd.read_csv("/content/desc.csv")
cats = pd.read_csv("/content/merged_category.csv")

In [ ]:
desc

,cleaned_text
0,a significant problem with the centralized acc...
1,well. request detailed information about the c...
2,. request clarification about the billing and ...
3,well. ask about the compatibility of your prod...
4,in good health. i am eager to learn more about...
...,...
25042,there seems to be a discrepancy in my billing ...
25043,request a refund for the recent charge.
25044,twofactor authentication codes are not being d...
25045,i am unable to access my account after enterin...


In [ ]:
cats

,merged_category
0,System Performance & Availability
1,System Configuration & Integration
2,"Billing, Payments & Subscriptions"
3,Marketing & Digital Strategy
4,System Configuration & Integration
...,...
25042,"Billing, Payments & Subscriptions"
25043,"Billing, Payments & Subscriptions"
25044,System Performance & Availability
25045,System Performance & Availability


In [ ]:
desc['category'] = cats["merged_category"]

In [ ]:
desc.rename(columns = {'cleaned_text': 'description'}, inplace = True)

In [ ]:
desc.to_csv("desc_cats.csv", index = False)

In [ ]:
descats = pd.read_csv('/content/desc_cats.csv')

In [ ]:
descats[272:273]

,description,category
272,we are encountering a failure in vpnrouter con...,System Performance & Availability


In [ ]:
from sklearn.neural_network import MLPRegressor
import joblib


surrogate = MLPRegressor(
    hidden_layer_sizes=(256, 128),
    activation='relu',
    max_iter=200,
    random_state=42,
    verbose=True
)
surrogate.fit(embeddings_cpu, reduced_embeddings_cpu)


sample = surrogate.predict(embeddings_cpu[:5])
print("Output shape:", sample.shape)

joblib.dump(surrogate, "umap_surrogate.joblib")
print("Surrogate saved")

Iteration 1, loss = 2.04056287
Iteration 2, loss = 0.44270492
Iteration 3, loss = 0.32296327
Iteration 4, loss = 0.26782569
Iteration 5, loss = 0.23376045
Iteration 6, loss = 0.20964478
Iteration 7, loss = 0.18922636
Iteration 8, loss = 0.17243937
Iteration 9, loss = 0.15759495
Iteration 10, loss = 0.14637116
Iteration 11, loss = 0.13504747
Iteration 12, loss = 0.12439672
Iteration 13, loss = 0.11801883
Iteration 14, loss = 0.10960852
Iteration 15, loss = 0.10160699
Iteration 16, loss = 0.09572732
Iteration 17, loss = 0.08986752
Iteration 18, loss = 0.08557455
Iteration 19, loss = 0.08019896
Iteration 20, loss = 0.07556412
Iteration 21, loss = 0.07296162
Iteration 22, loss = 0.06838433
Iteration 23, loss = 0.06439127
Iteration 24, loss = 0.06177592
Iteration 25, loss = 0.05884020
Iteration 26, loss = 0.05659054
Iteration 27, loss = 0.05430194
Iteration 28, loss = 0.05234285
Iteration 29, loss = 0.04836779
Iteration 30, loss = 0.04667997
Iteration 31, loss = 0.04495201
Iteration 32, los